In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/playground-series-s5e10/sample_submission.csv
/kaggle/input/playground-series-s5e10/train.csv
/kaggle/input/playground-series-s5e10/test.csv


In [3]:
from sklearn.compose import make_column_selector, ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor

In [4]:
data = pd.read_csv('/kaggle/input/playground-series-s5e10/train.csv')
data.drop(['id'], inplace=True, axis=1)

In [6]:
data.head()

,road_type,num_lanes,curvature,speed_limit,lighting,weather,road_signs_present,public_road,time_of_day,holiday,school_season,num_reported_accidents,accident_risk
0,urban,2,0.06,35,daylight,rainy,False,True,afternoon,False,True,1,0.13
1,urban,4,0.99,35,daylight,clear,True,False,evening,True,True,0,0.35
2,rural,4,0.63,70,dim,clear,False,True,morning,True,False,2,0.30
3,highway,4,0.07,35,dim,rainy,True,True,morning,False,False,1,0.21
4,rural,1,0.58,60,daylight,foggy,False,False,evening,True,False,1,0.56


In [7]:
data.isnull().sum()

road_type                 0
num_lanes                 0
curvature                 0
speed_limit               0
lighting                  0
weather                   0
road_signs_present        0
public_road               0
time_of_day               0
holiday                   0
school_season             0
num_reported_accidents    0
accident_risk             0
dtype: int64

In [8]:
num_cols = make_column_selector(dtype_include='number')
cat_cols = make_column_selector(dtype_include=['object', 'bool'])

In [9]:
data[num_cols].columns

Index(['num_lanes', 'curvature', 'speed_limit', 'num_reported_accidents',
       'accident_risk'],
      dtype='object')

In [10]:
data[cat_cols].columns

Index(['road_type', 'lighting', 'weather', 'road_signs_present', 'public_road',
       'time_of_day', 'holiday', 'school_season'],
      dtype='object')

In [11]:
X = data.drop(['accident_risk'], axis=1)
y = data['accident_risk'].values

In [12]:
cat_pipe = Pipeline([
    ('cat', OneHotEncoder())
])

num_pipe = Pipeline([
    ('num', StandardScaler())
])

ctx = ColumnTransformer([
    ('cat', cat_pipe, cat_cols),
    ('num', num_pipe, num_cols)
], remainder='passthrough')

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train = ctx.fit_transform(X_train)
X_test = ctx.transform(X_test)

In [13]:
'''
lgbm = LGBMRegressor(objective='binary')
lgbm.fit(X_train, y_train)
y_pred = lgbm.predict(X_test)

print('MSE for lgbm:', mean_squared_error(y_test, y_pred))
print('R2 for lgbm:', r2_score(y_test, y_pred))
'''

"\nlgbm = LGBMRegressor(objective='binary')\nlgbm.fit(X_train, y_train)\ny_pred = lgbm.predict(X_test)\n\nprint('MSE for lgbm:', mean_squared_error(y_test, y_pred))\nprint('R2 for lgbm:', r2_score(y_test, y_pred))\n"

In [14]:
cat = CatBoostRegressor(verbose=0)
cat.fit(X_train, y_train)
y_pred = cat.predict(X_test)

print('MSE for catboost:', mean_squared_error(y_test, y_pred))
print('R2 for catboost:', r2_score(y_test, y_pred))

MSE for catboost: 0.0031612873982016193
R2 for catboost: 0.8855111850975814


In [15]:
param_distributions = {
    'iterations': [200, 400, 800, 1200, 1600],
    'learning_rate': [0.01, 0.03, 0.05, 0.1, 0.2],
    'depth': [4, 5, 6, 7, 8, 9, 10],
    'l2_leaf_reg': [1, 3, 5, 7, 9, 11, 13],
    'bagging_temperature': [0, 0.25, 0.5, 0.75, 1.0],
    'border_count': [32, 64, 128, 254],
    'random_strength': [0.5, 1, 2, 5, 10],
    'leaf_estimation_iterations': [1, 3, 5, 7, 9],
    'grow_policy': ['SymmetricTree', 'Depthwise', 'Lossguide'],
    'bootstrap_type': ['Bayesian', 'Bernoulli', 'MVS'],
    'subsample': [0.6, 0.8, 1.0],
    'min_data_in_leaf': [1, 5, 10, 20, 50, 100]
}

randomCv = RandomizedSearchCV(
    estimator=CatBoostRegressor(verbose=0),
    param_distributions=param_distributions,
    cv = 5,
    n_iter=50,
    scoring='neg_mean_squared_error',
    random_state=42,
    n_jobs=-1
)

ctx2 = ColumnTransformer([
    ('cat', cat_pipe, cat_cols),
    ('num', num_pipe, num_cols)
], remainder='passthrough')

X_transformed = ctx2.fit_transform(X)
gcv = randomCv.fit(X_transformed, y, verbose=1)

0:	learn: 0.1621733	total: 243ms	remaining: 4m 50s
1:	learn: 0.1580769	total: 389ms	remaining: 3m 53s
2:	learn: 0.1541204	total: 517ms	remaining: 3m 26s
3:	learn: 0.1503055	total: 590ms	remaining: 2m 56s
4:	learn: 0.1466196	total: 710ms	remaining: 2m 49s
5:	learn: 0.1430455	total: 836ms	remaining: 2m 46s
6:	learn: 0.1396143	total: 925ms	remaining: 2m 37s
7:	learn: 0.1362958	total: 1.05s	remaining: 2m 36s
8:	learn: 0.1330931	total: 1.19s	remaining: 2m 37s
9:	learn: 0.1300058	total: 1.32s	remaining: 2m 37s
10:	learn: 0.1270236	total: 1.47s	remaining: 2m 38s
11:	learn: 0.1241467	total: 1.63s	remaining: 2m 40s
12:	learn: 0.1213781	total: 1.81s	remaining: 2m 45s
13:	learn: 0.1187078	total: 2.01s	remaining: 2m 50s
14:	learn: 0.1161237	total: 2.17s	remaining: 2m 51s
15:	learn: 0.1136384	total: 2.27s	remaining: 2m 47s
16:	learn: 0.1112457	total: 2.37s	remaining: 2m 45s
17:	learn: 0.1089434	total: 2.56s	remaining: 2m 47s
18:	learn: 0.1067241	total: 2.73s	remaining: 2m 49s
19:	learn: 0.1045918	t

/usr/local/lib/python3.11/dist-packages/sklearn/model_selection/_validation.py:378: FitFailedWarning: 
160 fits failed out of a total of 250.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
90 fits failed with the following error:
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/sklearn/model_selection/_validation.py", line 686, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/usr/local/lib/python3.11/dist-packages/catboost/core.py", line 5873, in fit
    return self._fit(X, y, cat_features, text_features, embedding_features, None, graph, sample_weight, None, None, None, None, baseline,
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

0:	learn: 0.1527704	total: 58.4ms	remaining: 46.6s
1:	learn: 0.1403452	total: 108ms	remaining: 43s
2:	learn: 0.1301250	total: 144ms	remaining: 38.2s
3:	learn: 0.1199464	total: 190ms	remaining: 37.7s
4:	learn: 0.1110023	total: 234ms	remaining: 37.2s
5:	learn: 0.1031657	total: 281ms	remaining: 37.1s
6:	learn: 0.0963513	total: 325ms	remaining: 36.8s
7:	learn: 0.0904094	total: 371ms	remaining: 36.7s
8:	learn: 0.0852463	total: 414ms	remaining: 36.4s
9:	learn: 0.0809817	total: 460ms	remaining: 36.4s
10:	learn: 0.0771621	total: 506ms	remaining: 36.3s
11:	learn: 0.0740856	total: 546ms	remaining: 35.8s
12:	learn: 0.0712931	total: 590ms	remaining: 35.7s
13:	learn: 0.0688955	total: 634ms	remaining: 35.6s
14:	learn: 0.0670191	total: 677ms	remaining: 35.4s
15:	learn: 0.0653781	total: 725ms	remaining: 35.5s
16:	learn: 0.0640405	total: 771ms	remaining: 35.5s
17:	learn: 0.0628329	total: 813ms	remaining: 35.3s
18:	learn: 0.0619281	total: 855ms	remaining: 35.1s
19:	learn: 0.0610427	total: 898ms	remainin

In [16]:
print('best params:', gcv.best_params_)
print('best score:', gcv.best_score_)

best params: {'subsample': 1.0, 'random_strength': 5, 'min_data_in_leaf': 10, 'learning_rate': 0.1, 'leaf_estimation_iterations': 1, 'l2_leaf_reg': 5, 'iterations': 800, 'grow_policy': 'SymmetricTree', 'depth': 7, 'border_count': 128, 'bootstrap_type': 'MVS', 'bagging_temperature': 0.5}
best score: -0.003144248898098992
1571:	learn: 0.0563165	total: 6m 23s	remaining: 6.82s
1572:	learn: 0.0563162	total: 6m 23s	remaining: 6.58s
1573:	learn: 0.0563161	total: 6m 23s	remaining: 6.34s
1574:	learn: 0.0563158	total: 6m 23s	remaining: 6.09s
1575:	learn: 0.0563155	total: 6m 24s	remaining: 5.85s
1576:	learn: 0.0563153	total: 6m 24s	remaining: 5.61s
1577:	learn: 0.0563152	total: 6m 24s	remaining: 5.36s
1578:	learn: 0.0563150	total: 6m 24s	remaining: 5.12s
1579:	learn: 0.0563148	total: 6m 25s	remaining: 4.87s
1580:	learn: 0.0563146	total: 6m 25s	remaining: 4.63s
1581:	learn: 0.0563144	total: 6m 25s	remaining: 4.39s
1582:	learn: 0.0563142	total: 6m 25s	remaining: 4.14s
1583:	learn: 0.0563140	total: 

In [17]:
cat = CatBoostRegressor(
    subsample=1.0,
    random_strength=5,
    min_data_in_leaf=10,
    learning_rate=0.1,
    leaf_estimation_iterations=1,
    l2_leaf_reg=5,
    iterations=800,
    grow_policy='SymmetricTree',
    depth=7,
    border_count=128,
    bootstrap_type='MVS',
    bagging_temperature=0.5
)
cat.fit(X_train, y_train)
y_pred = cat.predict(X_test)

print('MSE for catboost:', mean_squared_error(y_test, y_pred))
print('R2 for catboost:', r2_score(y_test, y_pred))

0:	learn: 0.1528891	total: 54.6ms	remaining: 43.6s
1:	learn: 0.1408864	total: 96.7ms	remaining: 38.6s
2:	learn: 0.1306697	total: 129ms	remaining: 34.1s
3:	learn: 0.1204629	total: 167ms	remaining: 33.3s
4:	learn: 0.1114676	total: 206ms	remaining: 32.7s
5:	learn: 0.1035307	total: 251ms	remaining: 33.2s
6:	learn: 0.0966303	total: 291ms	remaining: 33s
7:	learn: 0.0906715	total: 326ms	remaining: 32.3s
8:	learn: 0.0854478	total: 361ms	remaining: 31.7s
9:	learn: 0.0814030	total: 399ms	remaining: 31.5s
10:	learn: 0.0777586	total: 445ms	remaining: 31.9s
11:	learn: 0.0745660	total: 476ms	remaining: 31.3s
12:	learn: 0.0717214	total: 516ms	remaining: 31.2s
13:	learn: 0.0692291	total: 553ms	remaining: 31s
14:	learn: 0.0671523	total: 589ms	remaining: 30.8s
15:	learn: 0.0655102	total: 626ms	remaining: 30.7s
16:	learn: 0.0641128	total: 665ms	remaining: 30.6s
17:	learn: 0.0628788	total: 698ms	remaining: 30.3s
18:	learn: 0.0619514	total: 735ms	remaining: 30.2s
19:	learn: 0.0611610	total: 772ms	remaining

In [18]:
test_data = pd.read_csv('/kaggle/input/playground-series-s5e10/test.csv')
test_data

,id,road_type,num_lanes,curvature,speed_limit,lighting,weather,road_signs_present,public_road,time_of_day,holiday,school_season,num_reported_accidents
0,517754,highway,2,0.34,45,night,clear,True,True,afternoon,True,True,1
1,517755,urban,3,0.04,45,dim,foggy,True,False,afternoon,True,False,0
2,517756,urban,2,0.59,35,dim,clear,True,False,afternoon,True,True,1
3,517757,rural,4,0.95,35,daylight,rainy,False,False,afternoon,False,False,2
4,517758,highway,2,0.86,35,daylight,clear,True,False,evening,False,True,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...
172580,690334,rural,2,0.01,45,dim,rainy,False,False,afternoon,True,True,2
172581,690335,rural,1,0.74,70,daylight,foggy,False,True,afternoon,False,False,2
172582,690336,urban,2,0.14,70,dim,clear,False,False,evening,True,True,1
172583,690337,urban,1,0.09,45,daylight,foggy,True,True,morning,False,True,0


In [20]:
ids = test_data['id']
Xt_data = test_data.drop(['id'], axis=1)
Xt_data = ctx.transform(Xt_data)

In [21]:
yt_pred = cat.predict(Xt_data)
np.savetxt('output.csv', np.column_stack((ids, yt_pred)), delimiter=',', comments='', header='id,accident_risk', fmt=['%d', '%.2f'])

In [127]:
rf = RandomForestRegressor()
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

print('MSE for catboost:', mean_squared_error(y_test, y_pred))
print('R2 for catboost:', r2_score(y_test, y_pred))

MSE for catboost: 0.0035317500530205807
R2 for catboost: 0.8720945528926276


In [128]:
yt_pred = rf.predict(Xt_data)
np.savetxt('output2.csv', np.column_stack((ids, yt_pred)), delimiter=',', comments='', header='id,accident_risk', fmt=['%d', '%.2f'])